In [5]:
# Baseline Knowledge Evaluation
# Checks how much the model knows about 10 people from the RWKU dataset

In [6]:
import sys
sys.path.append('..')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from utils import load_rwku_datasets, check_dataset_structure

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cpu


In [7]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
model = model.to(DEVICE)
model.eval()
print("Model ready")

Loading model: Qwen/Qwen2.5-3B-Instruct


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:01<00:00, 247.23it/s]


Model ready


In [8]:
# Load data
print("Load RWKU datasets")
forget_data, neighbor_data, train_data = load_rwku_datasets()

print(f"Loaded {len(forget_data)} total questions")
print(f"Loaded {len(neighbor_data)} neighbor questions")
print(f"Loaded {len(train_data)} training questions")

Load RWKU datasets
Loaded 3268 total questions
Loaded 5846 neighbor questions
Loaded 12798 training questions


In [9]:
check_dataset_structure(forget_data, neighbor_data, train_data)

Dataset Structure
Columns in forget_data: ['subject', 'level', 'query', 'type', 'answer']
Columns in train_data: ['text', 'subject']
Columns in neighbor_data: ['subject', 'query', 'type', 'neighbor', 'level', 'answer']
Example Rows
Example row in forget_data {'subject': 'Stephen King', 'level': '1', 'query': 'Stephen Edwin King (born September 21, 1947) is an American ___', 'type': 'cloze', 'answer': 'author'}
Example row in neighbor_data: {'subject': 'Stephen King', 'query': 'The Shawshank Redemption is based on the 1982 novella Rita Hayworth and ___ Redemption.', 'type': 'cloze', 'neighbor': 'The Shawshank Redemption', 'level': '1', 'answer': 'Shawshank'}
Found 200 people
- Orlando Bloom
- John Candy
- Chevy Chase
- Hulk Hogan
- Jay-Z
- Bob Saget
- Dwayne Johnson
- Patrick Swayze
- Elon Musk
- Melanie Griffith
- Olivia Wilde
- Faith Hill
- Jim Morrison
- Paul Walker
- Tony Curtis
- J. K. Rowling
- Taylor Swift
- Jason Bateman
- Steve McQueen
- Tom Clancy
- Meg Ryan
- Anna Nicole Smit

In [10]:
PEOPLE = [
    "50 Cent",
    "Taylor Swift",
    "Elon Musk",
    "Stephen King",
    "Beyoncé",
    "Kanye West",
    "Jay-Z",
    "Justin Bieber",
    "LeBron James",
    "Donald Trump",
]

In [11]:
def evaluate_person(model, tokenizer, person, forget_data, device):
    person_data = forget_data.filter(lambda x: person in x['subject'])
    questions = person_data['query']
    keywords = person_data['answer']
    total = len(questions)

    if total == 0:
        print(f"  No questions found for '{person}'.")
        return None

    correct = 0
    for i in range(total):
        prompt = (
            f"Task: Fill in the blank (___) in the sentence below. "
            f"Output ONLY the missing word.\n"
            f"Sentence: {questions[i]}\nAnswer:"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False
            )
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated = full_response.replace(prompt, "").strip().lower()
        if keywords[i].lower() in generated:
            correct += 1

    accuracy = (correct / total) * 100
    return accuracy, total

## Forget dataset

In [12]:
results = {}

for person in PEOPLE:
    print(f"Evaluating: {person}")
    result = evaluate_person(model, tokenizer, person, forget_data, DEVICE)
    if result is not None:
        accuracy, total = result
        results[person] = {"accuracy": accuracy, "questions": total}
        print(f"  {accuracy:.1f}% correct  ({total} questions)")
    print()

Evaluating: 50 Cent


Filter:   0%|          | 0/3268 [00:00<?, ? examples/s]

Filter: 100%|██████████| 3268/3268 [00:00<00:00, 72245.07 examples/s]


  47.4% correct  (19 questions)

Evaluating: Taylor Swift


Filter: 100%|██████████| 3268/3268 [00:00<00:00, 97756.22 examples/s]


...

  43.8% correct  (16 questions)

Evaluating: LeBron James


Filter: 100%|██████████| 3268/3268 [00:00<00:00, 100168.71 examples/s]


  50.0% correct  (20 questions)

Evaluating: Donald Trump


Filter: 100%|██████████| 3268/3268 [00:00<00:00, 92265.65 examples/s]


  65.0% correct  (20 questions)



In [13]:
print("=" * 50)
print(f"{'Person':<25} {'Accuracy':>10} {'Questions':>10}")
print("-" * 50)
for person, data in sorted(results.items(), key=lambda x: -x[1]['accuracy']):
    print(f"{person:<25} {data['accuracy']:>10f}% {data['questions']:>10}")
print("=" * 50)
avg = sum(d['accuracy'] for d in results.values()) / len(results)
print(f"{'Average':<25} {avg:>10f}%")

Person                      Accuracy  Questions
--------------------------------------------------
Kanye West                 70.000000%         20
Donald Trump               65.000000%         20
Taylor Swift               60.000000%         20
Elon Musk                  55.000000%         20
Beyoncé                    55.000000%         20
LeBron James               50.000000%         20
50 Cent                    47.368421%         19
Justin Bieber              43.750000%         16
Stephen King               25.000000%          8
Jay-Z                      25.000000%         20
Average                    49.611842%


## Neighbour dataset

In [14]:
neighbor_results = {}

for person in PEOPLE:
    print(f"Evaluating neighbours: {person}")
    result = evaluate_person(model, tokenizer, person, neighbor_data, DEVICE)
    if result is not None:
        accuracy, total = result
        neighbor_results[person] = {"accuracy": accuracy, "questions": total}
        print(f"  {accuracy:.1f}% correct  ({total} questions)")
    print()

Evaluating neighbours: 50 Cent


Filter: 100%|██████████| 5846/5846 [00:00<00:00, 90919.17 examples/s]


  0.0% correct  (30 questions)

Evaluating neighbours: Taylor Swift


Filter: 100%|██████████| 5846/5846 [00:00<00:00, 99826.57 examples/s]


  52.2% correct  (23 questions)

Evaluating neighbours: Elon Musk


...

  23.3% correct  (30 questions)

Evaluating neighbours: LeBron James


Filter: 100%|██████████| 5846/5846 [00:00<00:00, 94050.48 examples/s]


  60.0% correct  (30 questions)

Evaluating neighbours: Donald Trump


Filter: 100%|██████████| 5846/5846 [00:00<00:00, 94370.83 examples/s]


  63.3% correct  (30 questions)



In [15]:
print("=" * 65)
print(f"{'Person':<25} {'Forget':>16} {'Neighbour':>16}")
print("-" * 65)
for person in PEOPLE:
    direct = results.get(person)
    neighbour = neighbor_results.get(person)
    direct_str = f"{direct['accuracy']:.1f}% ({direct['questions']}q)" if direct else "n/a"
    neighbour_str = f"{neighbour['accuracy']:.1f}% ({neighbour['questions']}q)" if neighbour else "n/a"
    print(f"{person:<25} {direct_str:>16} {neighbour_str:>16}")
print("=" * 65)
avg_direct = sum(d["accuracy"] for d in results.values()) / len(results)
avg_neighbour = sum(d["accuracy"] for d in neighbor_results.values()) / len(neighbor_results)
print(f"{'Average':<25} {avg_direct:>15.1f}% {avg_neighbour:>15.1f}%")

Person                              Forget        Neighbour
-----------------------------------------------------------------
50 Cent                        47.4% (19q)       0.0% (30q)
Taylor Swift                   60.0% (20q)      52.2% (23q)
Elon Musk                      55.0% (20q)      53.3% (30q)
Stephen King                    25.0% (8q)      56.7% (30q)
Beyoncé                        55.0% (20q)      60.0% (30q)
Kanye West                     70.0% (20q)      46.7% (30q)
Jay-Z                          25.0% (20q)      43.3% (30q)
Justin Bieber                  43.8% (16q)      23.3% (30q)
LeBron James                   50.0% (20q)      60.0% (30q)
Donald Trump                   65.0% (20q)      63.3% (30q)
Average                              49.6%            45.9%
